# AI categorization — model comparison & full runs

Heavy logic lives in [`cr/categorize.py`](cr/categorize.py) and [`cr/prompts.py`](cr/prompts.py). Run Jupyter with **cwd = repo root** (`ecothrift-dashboard`). Requires `pip install -r workspace/notebooks/_shared/requirements-notebooks.txt` and `ANTHROPIC_API_KEY`.

## 1. Setup — Django + `cr` on path

In [1]:
import os
import sys
from pathlib import Path

nb_dir = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd().resolve()
REPO = nb_dir
while REPO != REPO.parent:
    if (REPO / "manage.py").is_file():
        break
    REPO = REPO.parent
else:
    raise RuntimeError("Could not find repo root (no manage.py)")

os.chdir(REPO)
CR_ROOT = REPO / "workspace" / "notebooks" / "category-research"
sys.path.insert(0, str(CR_ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "ecothrift.settings")

import django

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()
print("Django OK:", django.get_version())
print("CWD:", Path.cwd())

Django OK: 5.2
CWD: E:\ecothrift-dashboard


## 2. Parameters

In [2]:
source = "bin1"
n = 10
model = "claude-sonnet-4-20250514"  # options: claude-sonnet-4-20250514, claude-haiku-4-5-20251001
n_jobs = 16
chunk_size = 500  # rows per disk chunk for ai_categorize_full

In [ ]:
from cr import get_sample, ai_categorize

models = ["claude-sonnet-4-20250514", "claude-haiku-4-5-20251001", "claude-opus-4-20250514"]
data = get_sample(source, n)

show_cols = [
    "product_title", "product_brand", "manifest_category",
    "manifest_subcategory", "manifest_description",
    "vendor_name", "item_retail_amt", "ai_category", "ai_confidence",
]

for m in models:
    print(f"\n{'='*80}")
    print(f"MODEL: {m}")
    print(f"{'='*80}")
    results = ai_categorize(data.copy(), m, n_jobs)
    display(results[show_cols])

## 3. Sample + categorize (in-memory; compare models here)

In [ ]:
from IPython.display import display

from cr import ai_categorize, get_sample

data = get_sample(source, n)
results = ai_categorize(data, model, n_jobs)
display(results)

## 4. Full run (chunked, resumable)

In [3]:
# Cell 1: Bin 2 full categorization
source = "bin2"
model = "claude-sonnet-4-20250514"
n_jobs = 16
chunk_size = 500

from cr import ai_categorize_full
full_results_bin2 = ai_categorize_full(source, model, n_jobs, chunk_size=chunk_size)

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

...

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/307 [00:00<?, ?it/s]

In [4]:
# Cell 2: Bin 3 full categorization
source = "bin3"
model = "claude-sonnet-4-20250514"
n_jobs = 16
chunk_size = 500

full_results_bin3 = ai_categorize_full(source, model, n_jobs, chunk_size=chunk_size)

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

...

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/500 [00:00<?, ?it/s]

ai_categorize:   0%|          | 0/84 [00:00<?, ?it/s]

## 5. Concatenate all bins + summary (reads `*_categorized.csv`)

In [1]:
# Cell 3: Summary (run after all three bins are done)
from cr import build_summary
summary = build_summary()
display(summary.by_ai)
display(summary.by_manifest)

,ai_category,pct_bin1,pct_bin2,pct_bin3,avg_sale_price_bin2,avg_retail_bin2,avg_margin_bin2,avg_days_to_sell_bin2
9,Kitchen & dining,0.0,18.3933,18.1850,13.346667,27.866042,0.494660,None
18,Toys & games,0.0,13.0833,6.5471,14.156046,29.018985,0.482335,None
7,Home décor & lighting,0.0,9.2651,15.2610,12.057474,24.570089,0.497397,None
15,Sports & outdoors,0.0,9.2240,9.9720,21.512656,47.690875,0.465319,None
17,Tools & hardware,0.0,7.1986,9.6109,34.765703,76.587510,0.468737,None
10,Mixed lots & uncategorized,0.0,6.6375,0.1747,5.261485,12.201320,0.431248,None
6,"Health, beauty & personal care",0.0,5.8163,5.9879,12.671435,27.222188,0.490235,None
1,Baby & kids,0.0,5.6658,4.2870,12.853889,27.968043,0.466463,None
2,Bedding & bath,0.0,5.0636,5.0909,15.921135,33.705730,0.475394,None
8,Household & cleaning,0.0,3.8046,1.2931,11.737806,23.874245,0.504397,None


,manifest_category,pct_bin1,pct_bin2,pct_bin3,avg_sale_price_bin2,avg_retail_bin2,avg_margin_bin2,avg_days_to_sell_bin2
0,(blank),0.0,18.8449,19.0238,14.732215,31.797081,0.488256,None
141,KITCHEN_AND_DINING,0.0,16.3816,16.4376,8.489883,17.203158,0.500062,None
125,HOME_DECOR,0.0,6.8291,12.9077,10.946052,23.639038,0.485014,None
128,HOUSEHOLD_ESSENTIALS,0.0,5.5426,5.8364,10.420025,22.428099,0.477609,None
108,HARDWARE AND TOOLS,0.0,4.0372,3.5065,39.028644,86.765525,0.460655,None
...,...,...,...,...,...,...,...,...
9,Appliance Parts & Accessories,0.0,0.0000,0.0116,NaN,NaN,NaN,None
271,Vanities,0.0,0.0000,0.0233,NaN,NaN,NaN,None
2,ACCESSORIES,0.0,0.0000,0.0116,NaN,NaN,NaN,None
282,WOMENS_SHOES,0.0,0.0000,0.0116,NaN,NaN,NaN,None
